# 01 — Stage-1 Multi-Label Pathology Triage Training (Google Colab GPU / TPU)
Trains `MultiLabelTriageEnsemble` (`StrongCNN` + `ConvNeXtV2CoordinateTriage`) with `AsymmetricFocalLoss` on the preprocessed `stage1_triage_splits.csv` (`512x512`, 4,150 clean scans).
- **Target Pathologies:** Independent Sigmoid probabilities for `[Diabetic Retinopathy, Glaucoma, Cataract]`.
- **Automatic Checkpointing:** Saves `stage1_multi_label_best.pth` directly to Google Drive (`/content/drive/MyDrive/Retinopathy_Checkpoints/`) after every improved epoch.

In [ ]:
# 1. Mount Google Drive, Clone Repo & Unpack Preprocessed Dataset to Local Colab Disk
import os, sys, shutil, tarfile, zipfile
from pathlib import Path
import subprocess
import torch

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/Retinopathy')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/dprincepiriyan/Retinopathy.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    # Locate retinopathy_processed.tar.gz (or .zip) anywhere in Google Drive and extract to /content/data/processed
    LOCAL_DATA_ROOT = Path('/content/data/processed')
    if not (LOCAL_DATA_ROOT / 'stage1_triage_splits.csv').exists():
        drive_root = Path('/content/drive/MyDrive')
        candidates = list(drive_root.rglob('retinopathy_processed.tar.gz')) + list(drive_root.rglob('retinopathy_processed.zip'))
        if not candidates:
            raise FileNotFoundError(
                'Could not find retinopathy_processed.tar.gz in Google Drive! '
                'Please upload retinopathy_processed.tar.gz to your Google Drive first.'
            )
        archive_path = candidates[0]
        print(f'Extracting {archive_path} to /content/data/ ...')
        Path('/content/data').mkdir(parents=True, exist_ok=True)
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as zf:
                zf.extractall('/content/data')
        else:
            with tarfile.open(archive_path, 'r:gz') as tf:
                tf.extractall('/content/data')
        print('Extraction complete!')
    DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Retinopathy_Checkpoints')
else:
    REPO_DIR = Path.cwd().resolve()
    if REPO_DIR.name == 'colab':
        REPO_DIR = REPO_DIR.parent
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    LOCAL_DATA_ROOT = REPO_DIR / 'data' / 'processed'
    DRIVE_CKPT_DIR = REPO_DIR / 'checkpoints'

DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT_DIR = REPO_DIR / 'checkpoints'
LOCAL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

from src.utils.tpu_accelerator import ColabTPUAccelerator
accel = ColabTPUAccelerator(prefer_tpu=True)
print(f'Active Accelerator: {accel.info.device_name} | Backend: {accel.info.backend} | Precision: {accel.info.amp_dtype}')

In [ ]:
# 2. Build Patient-Stratified Stage-1 DataLoaders (512x512)
import numpy as np
from torch.utils.data import DataLoader
from src.data.datasets import PreprocessedManifestDataset

MANIFEST_CSV = str(LOCAL_DATA_ROOT / 'stage1_triage_splits.csv')
BATCH_SIZE = 16
VAL_FOLD = 4

train_ds = PreprocessedManifestDataset(
    manifest_csv=MANIFEST_CSV,
    split='train_val',
    exclude_fold=VAL_FOLD,
    resolution=512,
    data_root_override=str(LOCAL_DATA_ROOT),
    augment=True,
)
val_ds = PreprocessedManifestDataset(
    manifest_csv=MANIFEST_CSV,
    split='train_val',
    only_fold=VAL_FOLD,
    resolution=512,
    data_root_override=str(LOCAL_DATA_ROOT),
    augment=False,
)
test_ds = PreprocessedManifestDataset(
    manifest_csv=MANIFEST_CSV,
    split='test',
    resolution=512,
    data_root_override=str(LOCAL_DATA_ROOT),
    augment=False,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=(accel.info.backend == 'cuda'), drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=(accel.info.backend == 'cuda'), drop_last=False
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=(accel.info.backend == 'cuda'), drop_last=False
)

print(f'Stage-1 Train Scans (Folds 1-3): {len(train_ds)}')
print(f'Stage-1 Val Scans   (Fold {VAL_FOLD}):   {len(val_ds)}')
print(f'Stage-1 Test Scans  (Fold 0):   {len(test_ds)}')

In [ ]:
# 3. Initialize Stage-1 Multi-Label Triage Ensemble & Asymmetric Focal Loss
from sklearn.metrics import f1_score, roc_auc_score
from src.tier2_triage import MultiLabelTriageEnsemble
from src.losses import AsymmetricFocalLoss

EPOCHS = 15
LR = 3e-4

model = MultiLabelTriageEnsemble(base_channels=32).to(accel.device)
criterion = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05).to(accel.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def evaluate_stage1(loader):
    model.eval()
    all_probs, all_targets = [], []
    total_loss, n_batches = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            x = batch['image'].to(accel.device)
            y = batch['multi_label'].to(accel.device)
            with accel.autocast_context(enabled=True):
                out = model(x)
                loss = 0.45 * criterion(out['logits_strong'], y) + 0.55 * criterion(out['logits_convnext'], y)
            total_loss += float(loss.detach().cpu().item())
            n_batches += 1
            all_probs.append(out['probs_ensemble'].detach().float().cpu().numpy())
            all_targets.append(y.detach().float().cpu().numpy())

    probs_arr = np.concatenate(all_probs, axis=0)
    targets_arr = np.concatenate(all_targets, axis=0)
    preds_bin = (probs_arr >= 0.5).astype(np.int64)

    aucs = []
    for c in range(3):
        if len(np.unique(targets_arr[:, c])) > 1:
            aucs.append(float(roc_auc_score(targets_arr[:, c], probs_arr[:, c])))
        else:
            aucs.append(0.0)
    macro_auc = float(np.mean(aucs))
    macro_f1 = float(f1_score(targets_arr, preds_bin, average='macro', zero_division=0))
    return {
        'loss': total_loss / max(n_batches, 1),
        'macro_auc': macro_auc,
        'macro_f1': macro_f1,
        'auc_dr': aucs[0],
        'auc_glaucoma': aucs[1],
        'auc_cataract': aucs[2],
        'probs': probs_arr,
        'targets': targets_arr,
    }

In [ ]:
# 4. Execute Full Multi-Epoch Training Loop with Automatic Google Drive Checkpointing
import time

best_val_auc = -1.0
best_ckpt_local = LOCAL_CKPT_DIR / 'stage1_multi_label_best.pth'
best_ckpt_drive = DRIVE_CKPT_DIR / 'stage1_multi_label_best.pth'

print(f'Starting Stage-1 Training for {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, steps = 0.0, 0

    for batch in train_loader:
        x = batch['image'].to(accel.device)
        y = batch['multi_label'].to(accel.device)

        optimizer.zero_grad()
        with accel.autocast_context(enabled=True):
            out = model(x)
            loss = 0.45 * criterion(out['logits_strong'], y) + 0.55 * criterion(out['logits_convnext'], y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        accel.optimizer_step(optimizer)

        running_loss += float(loss.detach().cpu().item())
        steps += 1

    scheduler.step()
    train_loss = running_loss / max(steps, 1)
    val_metrics = evaluate_stage1(val_loader)
    elapsed = time.time() - t0

    improved = ''
    if val_metrics['macro_auc'] > best_val_auc:
        best_val_auc = val_metrics['macro_auc']
        accel.save_checkpoint(model, best_ckpt_local)
        if best_ckpt_drive != best_ckpt_local:
            shutil.copy2(best_ckpt_local, best_ckpt_drive)
        np.savez_compressed(
            DRIVE_CKPT_DIR / 'stage1_val_oof.npz',
            probs=val_metrics['probs'],
            targets=val_metrics['targets'],
        )
        improved = ' * [SAVED BEST TO DRIVE]'

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Macro-AUC: {val_metrics['macro_auc']:.4f} (DR:{val_metrics['auc_dr']:.3f}, "
        f"GL:{val_metrics['auc_glaucoma']:.3f}, CAT:{val_metrics['auc_cataract']:.3f}) | "
        f"Macro-F1: {val_metrics['macro_f1']:.4f}{improved}"
    )

In [ ]:
# 5. Final Evaluation on Untouched Test Split (Fold 0)
state_dict = torch.load(best_ckpt_local, map_location=accel.device, weights_only=True)
model.load_state_dict(state_dict)
test_metrics = evaluate_stage1(test_loader)

print('=== Stage-1 Untouched Test Split (Fold 0) Results ===')
print(f"Test Macro ROC-AUC : {test_metrics['macro_auc']:.4f}")
print(f"  - DR ROC-AUC       : {test_metrics['auc_dr']:.4f}")
print(f"  - Glaucoma ROC-AUC : {test_metrics['auc_glaucoma']:.4f}")
print(f"  - Cataract ROC-AUC : {test_metrics['auc_cataract']:.4f}")
print(f"Test Macro-F1      : {test_metrics['macro_f1']:.4f}")
print(f'Saved Checkpoint   : {best_ckpt_drive}')